In [9]:
#if fresh enviornment, in terminal
'''
chmod +x dependencies.sh
./dependencies.sh
'''
import sys
print(sys.executable)

/home/ec2-user/anaconda3/envs/python3/bin/python


In [10]:
#load dependacies
!pip install -q transformers datasets

import json


import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.model_selection import train_test_split
import re, os, math

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


Using device: cuda


In [16]:
#load data into df
#load the json training_set_task1.txt
with open("training_set_task1.txt", "r", encoding="utf-8") as f:
    data = json.load(f)

#load into pandas DataFrame
df = pd.DataFrame(data)

# Count using pandas (if 'labels' is a column containing lists)
empty_label_count = df["labels"].apply(lambda x: len(x) == 0).sum()

print(f"Number of entries with empty labels: {empty_label_count}")

#save entries with empty labels into a seperate DF for checking
empty_labels_df = df[df["labels"].map(len) == 0].copy()

# Keep only valid non-empty entries in your primary DataFrame
df = df[df["labels"].map(len) > 0].copy()

# Verify the row counts
print(f"Valid entries saved to df: {len(df)}")
print(f"Empty label entries saved to empty_labels_df: {len(empty_labels_df)}")


Number of entries with empty labels: 143
Valid entries saved to df: 545
Empty label entries saved to empty_labels_df: 143


In [ ]:

# Optional: If you are doing single-label classification, take the primary label
df["primary_label"] = df["labels"].apply(lambda x: x[0])

# 3. Label Mapping & Unique Classes
label_names = sorted(df["primary_label"].unique())
label_to_idx = {label: i for i, label in enumerate(label_names)}
NUM_CLASSES = len(label_names)

# 4. Extract texts and mapped integer labels
SUBSET_SIZE = 15000
texts = df["text"].tolist()
labels = [label_to_idx[l] for l in df["primary_label"]]

# Subsample if dataset exceeds SUBSET_SIZE
if SUBSET_SIZE < len(texts):
    texts, _, labels, _ = train_test_split(
        texts, labels, train_size=SUBSET_SIZE, random_state=42, stratify=labels
    )

# 5. Train/Val/Test Split (70/15/15)
X_temp, X_test, y_temp, y_test = train_test_split(
    texts, labels, test_size=0.15, random_state=42, stratify=labels
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42, stratify=y_temp
)

# Output summary
print(f"Classes: {label_names}")
print(f"Training:   {len(X_train)}")
print(f"Validation: {len(X_val)}")
print(f"Test:       {len(X_test)} (held out)")


word order carries meaning. Important